# Five-microgrid ablation study: trust-region samples vs. global random samples

This notebook runs one complete experiment from top to bottom. It does not load an old trust-region run. The four methods compared at the end are:

1. the original global L2O models solved with the NN solver;
2. trust-region residual correction initialized from the original NN solution;
3. global L2O models retrained on original + trust-region samples;
4. global L2O models retrained on original + equally many global random samples.

The original zero-initialized NN solution is loaded from `results/000`; the remaining run includes five lower-level models, a trust-region run, two rounds of training, and two new 10,000-step NN solves.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pandapower.networks as pn

import follower_solver  
from L2Otraining import TrainingConfig, train
from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.followers.ip import create_industrial_park_network
from stackelberg.instances import resolve_follower_configuration
from stackelberg.network import create_case33bw_20kv
from stackelberg.nn_solver import (
    NNSolverExperimentConfig,
    run_nn_solver_experiment,
)
from stackelberg.randomness import make_rng, set_global_seed
from stackelberg.trust_region import (
    TrustRegionExperimentConfig,
    run_trust_region_experiment,
)

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "NNmodel"
RESULTS_DIR = PROJECT_ROOT / "results"
ORIGINAL_NN_HISTORY_PATH = (
    RESULTS_DIR / "000" / "NN_solver_history_20260912-234954.csv"
)

if not (PROJECT_ROOT / "L2Otraining.py").is_file():
    raise RuntimeError("Run this notebook with the project root as the working directory.")

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

T = 24
SEED = 42
NN_STEPS = 10_000
NN_LEARNING_RATE = 1e-2
NN_SAVE_EVERY = 200
TRAINING_EPOCHS = 500
LOWER_TIME_LIMIT = 30.0
LOWER_MIP_GAP = 1e-3
NEW_DATA_CHECKPOINT_INTERVAL = 100
NEW_DATA_SEEDS = {
    "SB1": 4201,
    "SB2": 4202,
    "BG1": 4203,
    "BG2": 4204,
    "IP1": 4205,
}

SCENARIOS = ["SB1", "SB2", "BG1", "BG2", "IP1"]
MG_BUSES = [17, 32, 15, 21, 1]
UPPER_NET = create_case33bw_20kv()
LOWER_NETS = [
    pn.create_cigre_network_lv(),
    pn.create_cigre_network_lv(),
    create_cigre_residential_network(),
    create_cigre_residential_network(),
    create_industrial_park_network(),
]
X0 = np.full((len(SCENARIOS), T), 0.0, dtype=float)

follower_ids, scenario_types, follower_kwargs = resolve_follower_configuration(
    SCENARIOS
)
if follower_ids != SCENARIOS:
    raise RuntimeError(f"Resolved follower order {follower_ids} does not match {SCENARIOS}.")

INSTANCE_NETWORKS = dict(zip(SCENARIOS, LOWER_NETS))
INSTANCE_TYPES = dict(zip(SCENARIOS, scenario_types))
INSTANCE_SOLVER_KWARGS = dict(zip(SCENARIOS, follower_kwargs))

instance_table = pd.DataFrame(
    {
        "follower_id": SCENARIOS,
        "scenario_type": scenario_types,
        "mg_bus": MG_BUSES,
        "solver_kwargs": follower_kwargs,
    }
).set_index("follower_id")
display(instance_table)

,scenario_type,mg_bus,solver_kwargs
follower_id,,,
SB1,SB,17,{}
SB2,SB,32,{'load_scale': 1.1}
BG1,BG,15,{'date': '2023-01-01'}
BG2,BG,21,{'date': '2023-07-01'}
IP1,IP,1,{'date': '2023-01-01'}


In [2]:
def load_aligned_arrays(x_path, y_path, *, expected_rows=None, expected_width=T):
    """Load and validate one aligned price/response dataset."""
    x_path = Path(x_path)
    y_path = Path(y_path)
    if not x_path.is_file() or not y_path.is_file():
        raise FileNotFoundError(f"Expected both {x_path} and {y_path}.")

    x_data = np.load(x_path)
    y_data = np.load(y_path)
    if x_data.ndim != 2 or y_data.ndim != 2:
        raise ValueError(f"Expected 2-D arrays, got X{x_data.shape} and Y{y_data.shape}.")
    if x_data.shape != y_data.shape:
        raise ValueError(f"X/Y shape mismatch: X{x_data.shape} and Y{y_data.shape}.")
    if x_data.shape[1] != expected_width:
        raise ValueError(f"Expected width {expected_width}, got {x_data.shape[1]}.")
    if len(x_data) == 0:
        raise ValueError(f"Dataset is empty: {x_path.name}.")
    if expected_rows is not None and len(x_data) != expected_rows:
        raise ValueError(
            f"Expected {expected_rows} rows in {x_path.name}, got {len(x_data)}."
        )
    if not np.isfinite(x_data).all() or not np.isfinite(y_data).all():
        raise ValueError(f"Non-finite values found in {x_path.name} or {y_path.name}.")
    if np.any(x_data < -1e-12) or np.any(x_data > 1.5 + 1e-12):
        raise ValueError(f"Prices in {x_path.name} must lie in [0, 1.5].")
    return x_data, y_data


def atomic_save_npy(path, array):
    """Replace a NumPy file only after the complete array is serialized."""
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("wb") as file:
        np.save(file, np.asarray(array))
    temporary_path.replace(path)


for scenario in SCENARIOS:
    load_aligned_arrays(
        DATA_DIR / f"X_{scenario}.npy",
        DATA_DIR / f"Y_{scenario}.npy",
    )
    for suffix in (".pth", ".json"):
        model_path = MODEL_DIR / f"L2O_{scenario}{suffix}"
        if not model_path.is_file():
            raise FileNotFoundError(model_path)

print("Validated original datasets and L2O checkpoints for all five instances.")

Validated original datasets and L2O checkpoints for all five instances.


## 1. Load the existing zero-initialized original NN solution

The original NN solver is not rerun. Its last saved iterate is loaded from `results/000`; the two retrained NN-solver runs use the same all-zero initial price matrix and 10,000-step budget.

In [3]:
if not ORIGINAL_NN_HISTORY_PATH.is_file():
    raise FileNotFoundError(ORIGINAL_NN_HISTORY_PATH)

original_nn_history = pd.read_csv(ORIGINAL_NN_HISTORY_PATH)
if original_nn_history.empty:
    raise ValueError(f"Warm-start history is empty: {ORIGINAL_NN_HISTORY_PATH}")

price_columns = [f"pi_{index}" for index in range(len(SCENARIOS) * T)]
missing_columns = [
    column for column in ["iteration", "true_loss", *price_columns]
    if column not in original_nn_history.columns
]
if missing_columns:
    raise ValueError(f"Warm-start history is missing columns: {missing_columns}")

original_nn_last = original_nn_history.sort_values("iteration").iloc[-1]
original_nn_prices = original_nn_last[price_columns].to_numpy(dtype=float).reshape(
    len(SCENARIOS), T
)
if not np.isfinite(original_nn_prices).all():
    raise ValueError("Warm-start prices contain non-finite values.")
if np.any(original_nn_prices < 0.0) or np.any(original_nn_prices > 1.5):
    raise ValueError("Warm-start prices must lie in [0, 1.5].")
original_true_objective = float(original_nn_last["true_loss"])
if not np.isfinite(original_true_objective):
    raise ValueError("Warm-start true objective is not finite.")

original_nn_result = {
    "optimal pi": original_nn_prices,
    "true_obj": original_true_objective,
    "runtime_seconds": np.nan,
    "history_path": str(ORIGINAL_NN_HISTORY_PATH),
}

print("Loaded original NN warm start:", original_nn_result["history_path"])
print("Saved iteration:", int(original_nn_last["iteration"]))
print("Original NN true objective:", original_nn_result["true_obj"])
display(pd.DataFrame(original_nn_prices, index=SCENARIOS))

Loaded original NN warm start: /Users/liqi/Desktop/Codes/Surrogated-assisted Stackelnerg Game/results/000/NN_solver_history_20260912-234954.csv
Saved iteration: 9800
Original NN true objective: -103.73745796679384


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB1,0.905496,0.972395,0.992483,0.998625,0.982330,0.981594,1.006895,1.002889,0.991757,1.000325,...,1.037887,1.018056,1.024587,0.997190,0.993872,1.024512,1.423613,1.006008,1.086199,0.996892
SB2,0.984292,0.985921,0.971553,0.977133,0.962063,0.958349,1.004334,0.994941,1.498616,0.988460,...,1.011563,0.999690,0.996986,0.981880,1.001129,1.013935,0.991371,0.986417,1.089251,0.993309
BG1,1.499927,1.499906,1.499724,1.499684,1.499929,1.499869,1.499946,1.499939,1.499915,1.499948,...,1.499942,1.499932,1.499939,1.499932,1.499938,1.499940,1.499914,0.261827,1.499944,1.499898
BG2,1.499915,1.499862,1.499867,1.499870,1.499940,1.499835,1.499953,1.499933,1.499934,1.499927,...,1.499954,1.499947,1.499947,1.499948,1.499952,1.499953,1.499886,1.499871,1.499922,1.499468
IP1,1.493962,1.488898,1.495653,1.494458,1.490675,1.476853,1.496844,1.499135,1.499072,1.499319,...,1.499453,1.499741,1.499321,1.497359,1.494321,1.498004,1.496904,1.494693,1.498408,1.498219


## 2. Trust region and fresh trust-region training samples

The trust region starts at the original NN solution. `save_true_responses=True` retains the high-fidelity residual-fit designs, including each local center, and the controller removes duplicate rows across iterations before export. The resulting files replace the known-invalid `_tr` datasets.

In [4]:
tr_config = TrustRegionExperimentConfig(
    upper_net=UPPER_NET,
    lower_nets=LOWER_NETS,
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=SEED,
    device="cpu",
    x0=original_nn_prices,
    lower_time_limit=LOWER_TIME_LIMIT,
    lower_mip_gap=LOWER_MIP_GAP,
    copt_initialize_network_state=True,
    tr_params={
        "R_0": 0.05,
        "R_max": 1.0,
        "gamma_dec": 0.7,
        "gamma_inc": 1.5,
        "eta_1": 0.1,
        "eta_2": 0.75,
    },
    algo_params={
        "n_samples": 60,
        "K_max": 60,
        "sampling_method": "lhs",
        "residual_update_mode": "accumulate",
        "rbf_geometry_condition_max": 1e12,
        "solve_params": {
            "steps_init": 0,
            "steps_warm": 2000,
            "lr": 1e-2,
            "penalty_weights": {
                "eq": 1000.0,
                "ineq": 100.0,
                "soc": 100.0,
                "bound": 10.0,
            },
            "tr_weight": 1e6,
            "verbose": True,
        },
    },
    residual_model_params={
        "model_type": "rbf",
        "rbf_kernel": "cubic",
        "rbf_smoothing": 1e-3,
        "rbf_degree": 1,
        "rbf_scale": "trust_radius",
        "rbf_epsilon": None,
    },
    save_results=True,
    output_dir=str(RESULTS_DIR),
    save_true_responses=True,
    true_response_dir=str(DATA_DIR),
    true_response_tag="tr",
)
tr_result = run_trust_region_experiment(tr_config)
tr_prices = np.asarray(tr_result["x_opt"], dtype=float).reshape(
    len(SCENARIOS), T
)
tr_history = pd.DataFrame(tr_result["history"])
if tr_history.empty:
    raise RuntimeError("Trust-region history is empty.")
tr_last = tr_history.iloc[-1]
tr_objective = float(
    tr_last["F_candidate"]
    if str(tr_last["decision"]).startswith("accept")
    else tr_last["F_current"]
)
tr_composite_queries = int(
    tr_result["n_iter"] * (tr_config.algo_params["n_samples"] + 1)
)

print("Trust region converged:", tr_result["converged"])
print("Trust region iterations:", tr_result["n_iter"])
print("Trust region true objective:", tr_objective)
print("Trust region composite high-fidelity queries:", tr_composite_queries)
print("Saved paths:", tr_result.get("saved_paths"))

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Using global surrogate to solve initial point...
Initial surrogate solve used steps_init=0. Time taken to solve initial point 0.0613 seconds.
Iteration 0 | R=0.0500 
Building 60-point local designs for each of 5 followers (center + coordinate star + LHS) within R_tr=0.0500...
Fitting follower-local residual models with sample counts [60, 60, 60, 60, 60]...

In [5]:
current_tr_responses = tr_result.get("true_responses", {})
if set(current_tr_responses) != set(SCENARIOS):
    raise RuntimeError(
        f"Expected current-run TR samples for {SCENARIOS}, got "
        f"{sorted(current_tr_responses)}."
    )

tr_sample_counts = {}
sample_rows = []
for scenario in SCENARIOS:
    original_x, original_y = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}.npy",
        DATA_DIR / f"Y_{scenario}.npy",
    )
    tr_x, tr_y = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_tr.npy",
        DATA_DIR / f"Y_{scenario}_tr.npy",
    )
    in_memory_x = np.asarray(current_tr_responses[scenario]["X"], dtype=float)
    in_memory_y = np.asarray(current_tr_responses[scenario]["Y"], dtype=float)
    np.testing.assert_allclose(tr_x, in_memory_x)
    np.testing.assert_allclose(tr_y, in_memory_y)

    tr_sample_counts[scenario] = len(tr_x)
    sample_rows.append(
        {
            "scenario": scenario,
            "original_samples": len(original_x),
            "trust_region_samples": len(tr_x),
            "tr_combined_samples": len(original_x) + len(tr_x),
        }
    )

tr_sample_count_table = pd.DataFrame(sample_rows).set_index("scenario")
display(tr_sample_count_table)

,original_samples,trust_region_samples,tr_combined_samples
scenario,,,
SB1,10000,2748,12748
SB2,10000,2752,12752
BG1,10000,2646,12646
BG2,10000,1304,11304
IP1,10000,2682,12682


## 3. Retrain global models on original + trust-region samples

Each follower instance remains separate. Training writes `L2O_XX_tr.pth/.json` and leaves the original checkpoints unchanged.

In [6]:
def training_config(scenario):
    return TrainingConfig(
        scenario=scenario,
        hidden_dim=128,
        residual_blocks=3,
        batch_size=64,
        learning_rate=1e-3,
        epochs=TRAINING_EPOCHS,
        validation_fraction=0.2,
        seed=SEED,
    )


tr_training_results = {}
for scenario in SCENARIOS:
    print(f"\nTraining {scenario} on original + trust-region samples")
    tr_training_results[scenario] = train(
        scenario,
        config=training_config(scenario),
        data_tag="tr",
        model_tag="tr",
        include_original=True,
    )
    for suffix in (".pth", ".json"):
        expected_model = MODEL_DIR / f"L2O_{scenario}_tr{suffix}"
        if not expected_model.is_file():
            raise FileNotFoundError(expected_model)

display(pd.DataFrame(tr_training_results).T.rename_axis("scenario"))


Training SB1 on original + trust-region samples
SB1: concatenated original 10000 + tr 2748 samples -> 12748
SB1 epoch   10/500: train=0.050791, validation=0.053268
SB1 epoch   20/500: train=0.037227, validation=0.039422
SB1 epoch   30/500: train=0.028161, validation=0.029948
SB1 epoch   40/500: train=0.020378, validation=0.022005
SB1 epoch   50/500: train=0.012537, validation=0.013637
SB1 epoch   60/500: train=0.008957, validation=0.009988
SB1 epoch   70/500: train=0.009582, validation=0.010461
SB1 epoch   80/500: train=0.007296, validation=0.008062
SB1 epoch   90/500: train=0.007172, validation=0.007896
SB1 epoch  100/500: train=0.008777, validation=0.009383
SB1 epoch  110/500: train=0.004280, validation=0.004888
SB1 epoch  120/500: train=0.003644, validation=0.004159
SB1 epoch  130/500: train=0.002837, validation=0.003374
SB1 epoch  140/500: train=0.003291, validation=0.003715
SB1 epoch  150/500: train=0.008371, validation=0.008717
SB1 epoch  160/500: train=0.002032, validation=0.00

,train_mse,validation_mse
scenario,,
SB1,0.000446,0.000573
SB2,0.000771,0.000911
BG1,0.000983,0.001006
BG2,0.000535,0.000606
IP1,0.308006,0.706454


In [7]:
tr_nn_config = NNSolverExperimentConfig(
    upper_net=UPPER_NET,
    lower_nets=LOWER_NETS,
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=SEED,
    device="cpu",
    x0=X0,
    steps=NN_STEPS,
    lr=NN_LEARNING_RATE,
    save_every=NN_SAVE_EVERY,
    history_path=str(RESULTS_DIR / "NN_solver_tr_ablation_history.csv"),
    evaluate_true=True,
    save_results=True,
    model_tag="tr",
)
tr_nn_result = run_nn_solver_experiment(tr_nn_config)
tr_nn_prices = np.asarray(tr_nn_result["optimal pi"], dtype=float).reshape(
    len(SCENARIOS), T
)

print("TR-retrained NN history:", tr_nn_result["history_path"])
print("TR-retrained NN true objective:", tr_nn_result["true_obj"])
print("TR-retrained NN runtime (s):", tr_nn_result["runtime_seconds"])

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

[0] loss=6010.2324, obj=-1.1225, eq=6.0042, soc=0.0647, bound=0.0646, true_loss=124.0954
Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyr

## 4. Generate an equal-size global-random control dataset

For each follower instance, prices are drawn independently and uniformly from $[0,1.5]^{24}$. The requested sample count is the actual number of unique trust-region rows exported for that same instance. Existing `_new` arrays are never loaded or reused.

In [8]:
def generate_fresh_random_samples(
    scenario,
    n_samples,
    *,
    seed,
    checkpoint_interval=100,
    max_consecutive_failures=20,
):
    """Generate a fresh deterministic `_new` dataset for one instance."""
    if n_samples <= 0:
        raise ValueError("n_samples must be positive.")

    set_global_seed(seed)
    rng = make_rng(seed)
    solver_options = {
        **INSTANCE_SOLVER_KWARGS[scenario],
        "failure_mode": "none",
        "include_metadata": True,
        "time_limit": LOWER_TIME_LIMIT,
        "mip_gap": LOWER_MIP_GAP,
    }
    lower_solver = follower_solver.build_solver(
        INSTANCE_TYPES[scenario],
        T=T,
        net=INSTANCE_NETWORKS[scenario],
        **solver_options,
    )

    x_path = DATA_DIR / f"X_{scenario}_new.npy"
    y_path = DATA_DIR / f"Y_{scenario}_new.npy"
    x_data = np.empty((n_samples, T), dtype=float)
    y_data = np.empty((n_samples, T), dtype=float)
    completed = 0
    attempts = 0
    failures = 0
    consecutive_failures = 0
    checkpoint_start = time.time()

    print(f"{scenario}: generating {n_samples} fresh samples (seed={seed}).")
    while completed < n_samples:
        attempts += 1
        price = rng.uniform(0.0, 1.5, size=T)
        result = lower_solver.lower_solve(price)
        if result is None:
            failures += 1
            consecutive_failures += 1
            print(
                f"{scenario}: skipped failed attempt {attempts}; "
                f"collected {completed}/{n_samples}."
            )
            if consecutive_failures >= max_consecutive_failures:
                raise RuntimeError(
                    f"{scenario}: stopped after {consecutive_failures} "
                    "consecutive lower-level failures."
                )
            continue

        consecutive_failures = 0
        response = np.asarray(result["p_mg"], dtype=float).reshape(T)
        if not np.isfinite(response).all():
            raise ValueError(f"{scenario}: follower returned a non-finite response.")
        x_data[completed] = price
        y_data[completed] = response
        completed += 1

        if completed % checkpoint_interval == 0 or completed == n_samples:
            atomic_save_npy(x_path, x_data[:completed])
            atomic_save_npy(y_path, y_data[:completed])
            elapsed = time.time() - checkpoint_start
            print(
                f"{scenario}: saved {completed}/{n_samples} fresh samples "
                f"({failures} failed attempts, {elapsed:.1f} s)."
            )
            checkpoint_start = time.time()

    return x_data, y_data


In [9]:
new_data = {}
for scenario in SCENARIOS:
    new_data[scenario] = generate_fresh_random_samples(
        scenario,
        tr_sample_counts[scenario],
        seed=NEW_DATA_SEEDS[scenario],
        checkpoint_interval=NEW_DATA_CHECKPOINT_INTERVAL,
    )

sample_rows = []
for scenario in SCENARIOS:
    original_x, _ = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}.npy",
        DATA_DIR / f"Y_{scenario}.npy",
    )
    tr_x, _ = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_tr.npy",
        DATA_DIR / f"Y_{scenario}_tr.npy",
    )
    new_x, _ = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_new.npy",
        DATA_DIR / f"Y_{scenario}_new.npy",
        expected_rows=len(tr_x),
    )
    if len(new_x) != len(tr_x):
        raise AssertionError(
            f"{scenario}: new sample count {len(new_x)} != TR count {len(tr_x)}."
        )
    sample_rows.append(
        {
            "scenario": scenario,
            "original_samples": len(original_x),
            "trust_region_samples": len(tr_x),
            "new_random_samples": len(new_x),
            "tr_combined_samples": len(original_x) + len(tr_x),
            "new_combined_samples": len(original_x) + len(new_x),
        }
    )

sample_count_table = pd.DataFrame(sample_rows).set_index("scenario")
if not (
    sample_count_table["trust_region_samples"]
    == sample_count_table["new_random_samples"]
).all():
    raise AssertionError("Random and trust-region sample counts must match.")
display(sample_count_table)

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

SB1: generating 2748 fresh samples (seed=4201).
SB1: saved 100/2748 fresh samples (0 failed attempts, 1.5 s).
SB1: saved 200/2748 fresh samples (0 failed attempts, 1.5 s).
SB1: saved 300/2748 fresh samples (0 failed attempts, 1.5 s).
SB1: saved 400/2748 fresh samples (0 failed attempts, 1.5 s).
SB1: saved 500/2748 fresh samples (0 failed attempts, 1.5 s).
SB1: saved 600/2748 fresh samples (0 failed attempts, 1.6 s).
SB1: saved 700/2748 fresh samples (0 failed attempts, 1.6 s).
SB1: saved 800/2748 fresh samples (0 failed attempts, 1.5 s).
SB1: saved 900/2748 fresh samples (0 failed attempts, 1.5 s).
SB1: saved 1000/2748 fresh samples (0 failed attempts, 1.6 s).
SB1: saved 1100/2748 fresh samples (0 failed attempts, 2.1 s).
SB1: saved 1200/2748 fresh samples (0 failed attempts, 1.9 s).
SB1: saved 1300/2748 fresh samples (0 failed attempts, 1.8 s).
SB1: saved 1400/2748 fresh samples 

,original_samples,trust_region_samples,new_random_samples,tr_combined_samples,new_combined_samples
scenario,,,,,
SB1,10000,2748,2748,12748,12748
SB2,10000,2752,2752,12752,12752
BG1,10000,2646,2646,12646,12646
BG2,10000,1304,1304,11304,11304
IP1,10000,2682,2682,12682,12682


## 5. Retrain global models on original + equal-size random samples

Training writes `L2O_XX_new.pth/.json`, then the NN solver uses the same initial point and optimization budget as the other NN runs.

In [10]:
new_training_results = {}
for scenario in SCENARIOS:
    print(f"\nTraining {scenario} on original + equal-size global random samples")
    new_training_results[scenario] = train(
        scenario,
        config=training_config(scenario),
        data_tag="new",
        model_tag="new",
        include_original=True,
    )
    for suffix in (".pth", ".json"):
        expected_model = MODEL_DIR / f"L2O_{scenario}_new{suffix}"
        if not expected_model.is_file():
            raise FileNotFoundError(expected_model)

display(pd.DataFrame(new_training_results).T.rename_axis("scenario"))


Training SB1 on original + equal-size global random samples
SB1: concatenated original 10000 + new 2748 samples -> 12748
SB1 epoch   10/500: train=0.051246, validation=0.054013
SB1 epoch   20/500: train=0.033283, validation=0.035212
SB1 epoch   30/500: train=0.029868, validation=0.031879
SB1 epoch   40/500: train=0.020561, validation=0.022394
SB1 epoch   50/500: train=0.014672, validation=0.016130
SB1 epoch   60/500: train=0.011569, validation=0.012882
SB1 epoch   70/500: train=0.009107, validation=0.010411
SB1 epoch   80/500: train=0.007440, validation=0.008509
SB1 epoch   90/500: train=0.005540, validation=0.006441
SB1 epoch  100/500: train=0.004721, validation=0.005567
SB1 epoch  110/500: train=0.003828, validation=0.004506
SB1 epoch  120/500: train=0.002984, validation=0.003628
SB1 epoch  130/500: train=0.002445, validation=0.002980
SB1 epoch  140/500: train=0.002200, validation=0.002663
SB1 epoch  150/500: train=0.002326, validation=0.002755
SB1 epoch  160/500: train=0.001827, va

,train_mse,validation_mse
scenario,,
SB1,0.000387,0.000444
SB2,0.000450,0.000521
BG1,0.000919,0.000945
BG2,0.000440,0.000538
IP1,0.225030,0.687536


In [11]:
new_nn_config = NNSolverExperimentConfig(
    upper_net=UPPER_NET,
    lower_nets=LOWER_NETS,
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=SEED,
    device="cpu",
    x0=X0,
    steps=NN_STEPS,
    lr=NN_LEARNING_RATE,
    save_every=NN_SAVE_EVERY,
    history_path=str(RESULTS_DIR / "NN_solver_new_ablation_history.csv"),
    evaluate_true=True,
    save_results=True,
    model_tag="new",
)
new_nn_result = run_nn_solver_experiment(new_nn_config)
new_nn_prices = np.asarray(new_nn_result["optimal pi"], dtype=float).reshape(
    len(SCENARIOS), T
)

print("Random-retrained NN history:", new_nn_result["history_path"])
print("Random-retrained NN true objective:", new_nn_result["true_obj"])
print("Random-retrained NN runtime (s):", new_nn_result["runtime_seconds"])

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

[0] loss=6010.6826, obj=-1.1225, eq=6.0047, soc=0.0647, bound=0.0646, true_loss=124.0227
Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyr

## 6. Four-way solution and high-fidelity objective comparison

Lower true objective values are better. The key ablation comparison is between the two equally sized augmented training sets.

In [12]:
tr_saved_paths = tr_result.get("saved_paths", {})
comparison = pd.DataFrame(
    [
        {
            "method": "Original NN",
            "true_objective": float(original_nn_result["true_obj"]),
            "runtime_seconds": float(original_nn_result["runtime_seconds"]),
            "iterations": NN_STEPS,
            "composite_queries": np.nan,
            "history_or_result_path": original_nn_result["history_path"],
        },
        {
            "method": "Trust region",
            "true_objective": tr_objective,
            "runtime_seconds": float(tr_result["elapsed_seconds"]),
            "iterations": int(tr_result["n_iter"]),
            "composite_queries": tr_composite_queries,
            "history_or_result_path": tr_saved_paths.get("json"),
        },
        {
            "method": "NN: original + TR samples",
            "true_objective": float(tr_nn_result["true_obj"]),
            "runtime_seconds": float(tr_nn_result["runtime_seconds"]),
            "iterations": NN_STEPS,
            "composite_queries": np.nan,
            "history_or_result_path": tr_nn_result["history_path"],
        },
        {
            "method": "NN: original + equal global random samples",
            "true_objective": float(new_nn_result["true_obj"]),
            "runtime_seconds": float(new_nn_result["runtime_seconds"]),
            "iterations": NN_STEPS,
            "composite_queries": np.nan,
            "history_or_result_path": new_nn_result["history_path"],
        },
    ]
).set_index("method")

if len(comparison) != 4:
    raise AssertionError(f"Expected four comparison rows, got {len(comparison)}.")
if not np.isfinite(comparison["true_objective"].to_numpy(dtype=float)).all():
    raise ValueError("All four high-fidelity objectives must be finite.")

hour_columns = [f"hour_{hour:02d}" for hour in range(T)]
solution_frames = {
    "Original NN": pd.DataFrame(
        original_nn_prices, index=SCENARIOS, columns=hour_columns
    ),
    "Trust region": pd.DataFrame(
        tr_prices, index=SCENARIOS, columns=hour_columns
    ),
    "NN: original + TR samples": pd.DataFrame(
        tr_nn_prices, index=SCENARIOS, columns=hour_columns
    ),
    "NN: original + equal global random samples": pd.DataFrame(
        new_nn_prices, index=SCENARIOS, columns=hour_columns
    ),
}
solutions = pd.concat(solution_frames, names=["method", "scenario"])
if solutions.shape != (4 * len(SCENARIOS), T):
    raise AssertionError(f"Unexpected solution table shape: {solutions.shape}.")

output_timestamp = time.strftime("%Y%m%d-%H%M%S")
summary_path = RESULTS_DIR / f"retraining_ablation_summary_{output_timestamp}.csv"
solutions_path = RESULTS_DIR / f"retraining_ablation_solutions_{output_timestamp}.csv"
comparison.reset_index().to_csv(summary_path, index=False)
solutions.to_csv(solutions_path)

display(comparison)
display(sample_count_table)
display(solutions)
print("Saved objective summary to:", summary_path)
print("Saved four solution matrices to:", solutions_path)

,true_objective,runtime_seconds,iterations,composite_queries,history_or_result_path
method,,,,,
Original NN,-103.737458,NaN,10000,NaN,/Users/liqi/Desktop/Codes/Surrogated-assisted ...
Trust region,-105.941382,23405.220975,46,2806.0,/Users/liqi/Desktop/Codes/Surrogated-assisted ...
NN: original + TR samples,-88.731244,2166.634291,10000,NaN,/Users/liqi/Desktop/Codes/Surrogated-assisted ...
NN: original + equal global random samples,-100.188921,2217.241330,10000,NaN,/Users/liqi/Desktop/Codes/Surrogated-assisted ...


,original_samples,trust_region_samples,new_random_samples,tr_combined_samples,new_combined_samples
scenario,,,,,
SB1,10000,2748,2748,12748,12748
SB2,10000,2752,2752,12752,12752
BG1,10000,2646,2646,12646,12646
BG2,10000,1304,1304,11304,11304
IP1,10000,2682,2682,12682,12682


hour_00  \
method                                     scenario                 
Original NN                                SB1       9.054965e-01   
                                           SB2       9.842920e-01   
                                           BG1       1.499927e+00   
                                           BG2       1.499915e+00   
                                           IP1       1.493962e+00   
Trust region                               SB1       9.998795e-01   
                                           SB2       9.968669e-01   
                                           BG1       1.495644e+00   
                                           BG2       1.499982e+00   
                                           IP1       1.498066e+00   
NN: original + TR samples                  SB1       1.499928e+00   
                                           SB2       1.188536e-09   
                                           BG1       1.499981e+00   
                                           BG2       1.499069e+00   
                                           IP1       1.499439e+00   
NN: original + equal global random samples SB1       9.968435e-01   
                                           SB2       1.654597e-09   
                                           BG1       1.499954e+00   
                                           BG2       1.499952e+00   
                                           IP1       1.486897e+00   

                                                          hour_01   hour_02  \
method                                     scenario                           
Original NN                                SB1       9.723950e-01  0.992483   
                                           SB2       9.859210e-01  0.971553   
                                           BG1       1.499906e+00  1.499724   
                                           BG2       1.499862e+00  1.499867   
                                           IP1       1.488898e+00  1.495653   
Trust region                               SB1       9.979150e-01  0.999420   
                                           SB2       1.014724e+00  1.019202   
                                           BG1       1.488299e+00  1.451504   
                                           BG2       1.499983e+00  1.499984   
                                           IP1       1.493433e+00  1.497932   
NN: original + TR samples                  SB1       1.499951e+00  1.499834   
                                           SB2       4.943811e-10  0.000091   
                                           BG1       1.499914e+00  0.000073   
                                           BG2       1.499875e+00  1.499869   
                                           IP1       1.495921e+00  1.450716   
NN: original + equal global random samples SB1       9.813607e-01  1.007961   
                                           SB2       3.528125e-01  0.985318   
                                           BG1       1.499965e+00  1.499923   
                                           BG2       1.470373e+00  1.303163   
                                           IP1       1.434057e+00  1.470561   

                                                      hour_03   hour_04  \
method                                     scenario                       
Original NN                                SB1       0.998625  0.982330   
                                           SB2       0.977133  0.962063   
                                           BG1       1.499684  1.499929   
                                           BG2       1.499870  1.499940   
                                           IP1       1.494458  1.490675   
Trust region                               SB1       0.999871  0.999830   
                                           SB2       1.005840  0.994340   
                                           BG1       1.478333  1.494318   
                                           BG2    

Saved objective summary to: /Users/liqi/Desktop/Codes/Surrogated-assisted Stackelnerg Game/results/retraining_ablation_summary_20260925-204222.csv
Saved four solution matrices to: /Users/liqi/Desktop/Codes/Surrogated-assisted Stackelnerg Game/results/retraining_ablation_solutions_20260925-204222.csv
